# 04: Model Export & Backend Integration

In [1]:
import json
import os
import subprocess

import numpy as np

In [2]:
# Run this notebook from the `ai/` directory.
HERE = os.getcwd()
DATA_DIR = os.path.join(HERE, "data")
TRAINED_WEIGHTS = os.path.join(DATA_DIR, "trained_model_weights.json")
OUT_PATH = os.path.join(HERE, "model_weights.json")
SERVER_WEIGHTS = os.path.join(HERE, "..", "server", "src", "utils", "model_weights.json")

weights = json.load(open(TRAINED_WEIGHTS))
print("Features:", weights["features"])
print(f"Layers: {len(weights['layers'])} ->", [
    (len(l["weights"]), len(l["weights"][0]), l["activation"])
    for l in weights["layers"]
])

Features: ['txVolume', 'txFrequency', 'accountAge', 'networkDegree', 'timeRegularity', 'valueSentRatio', 'inOutRatio', 'degreeConcentration', 'valueVolatility']
Layers: 3 -> [(9, 12, 'relu'), (12, 6, 'relu'), (6, 1, 'sigmoid')]


In [3]:
def forward_numpy(weights, x):
    """Feedforward that mirrors server/src/utils/SimpleNeuralNetwork.js exactly."""
    a = np.asarray(x, dtype=float)
    for layer in weights["layers"]:
        W = np.asarray(layer["weights"], dtype=float)   # (in, out)
        b = np.asarray(layer["biases"], dtype=float)    # (out,)
        z = a @ W + b
        if layer["activation"] == "relu":
            a = np.maximum(z, 0.0)
        elif layer["activation"] == "sigmoid":
            a = 1.0 / (1.0 + np.exp(-z))
        elif layer["activation"] == "tanh":
            a = np.tanh(z)
        else:  # identity
            a = z
    return float(np.asarray(a).ravel()[0])

In [4]:
def feature_probe():
    def f(volume, freq, age_days, degree, gap_min, sent_ratio, inout, conc, max_val):
        return [
            np.tanh(np.log1p(volume) / 3.0),
            np.tanh(freq / 50.0),
            np.tanh(age_days / 365.0),
            np.tanh(degree / 20.0),
            np.tanh(np.log1p(gap_min) / 6.0),
            sent_ratio,
            inout,
            conc,
            np.tanh(np.log1p(max_val) / 3.0),
        ]

    return {
        "established active trader": f(3.0, 400, 800, 120, 900, 0.5, 0.5, 0.3, 50),
        "normal user": f(1.5, 40, 150, 12, 60, 0.5, 0.5, 0.3, 8),
        "fresh single tx": f(0.05, 1, 0.1, 1, 0, 1.0, 0.0, 1.0, 0.05),
        "spam flooder": f(0.1, 300, 2, 2, 0.2, 0.8, 0.5, 0.007, 0.1),
    }

print("Probe (low P(fraud) expected for established, high for fresh/spam):")
for name, vec in feature_probe().items():
    p = forward_numpy(weights, vec)
    print(f"  {name:<34} trust={1 - p:.3f}  (P(fraud)={p:.3f})")

Probe (low P(fraud) expected for established, high for fresh/spam):
  established active trader          trust=0.994  (P(fraud)=0.006)
  normal user                        trust=0.948  (P(fraud)=0.052)
  fresh single tx                    trust=0.846  (P(fraud)=0.154)
  spam flooder                       trust=0.716  (P(fraud)=0.284)


In [5]:
import shutil

shutil.copyfile(TRAINED_WEIGHTS, OUT_PATH)
shutil.copyfile(TRAINED_WEIGHTS, SERVER_WEIGHTS)
print(f"Exported {OUT_PATH} ({os.path.getsize(OUT_PATH):,} bytes)")
print(f"Exported {SERVER_WEIGHTS} ({os.path.getsize(SERVER_WEIGHTS):,} bytes)")

Exported /home/ayush/Desktop/code/SecureTransac/ai/model_weights.json (4,971 bytes)
Exported /home/ayush/Desktop/code/SecureTransac/ai/../server/src/utils/model_weights.json (4,971 bytes)


In [6]:
# Live cross-check: run the actual Node runtime on the same probe vectors.
JS_FILE = os.path.join(HERE, "..", "server", "src", "utils", "SimpleNeuralNetwork.js")
PROBES_FILE = os.path.join(DATA_DIR, "_probes.json")
NODE_SCRIPT = os.path.join(DATA_DIR, "_cross_check.mjs")

with open(PROBES_FILE, "w") as f:
    json.dump(feature_probe(), f)

node_src = """
import fs from 'fs';
import SimpleNeuralNetwork from '%s';
const weights = JSON.parse(fs.readFileSync('%s', 'utf-8'));
const vectors = JSON.parse(fs.readFileSync('%s', 'utf-8'));
const net = SimpleNeuralNetwork.fromJSON(weights);
for (const [name, vec] of Object.entries(vectors)) {
    const p = net.predict(vec);
    console.log(`%s`);
}
""" % (
    os.path.abspath(JS_FILE).replace("\\", "/"),
    TRAINED_WEIGHTS.replace("\\", "/"),
    PROBES_FILE.replace("\\", "/"),
    "${name}=${p.toFixed(6)}",
)

with open(NODE_SCRIPT, "w") as f:
    f.write(node_src)

result = subprocess.run(["node", NODE_SCRIPT], capture_output=True, text=True)
if result.returncode != 0:
    raise RuntimeError(f"Node cross-check failed:\n{result.stderr}")

node_probs = {}
for line in result.stdout.strip().splitlines():
    name, p = line.split("=")
    node_probs[name] = float(p)

max_diff = 0.0
print(f"{'probe':<34} {'python':>10} {'node':>10} {'diff':>10}")
for name, vec in feature_probe().items():
    py = forward_numpy(weights, vec)
    nd = node_probs[name]
    diff = abs(py - nd)
    max_diff = max(max_diff, diff)
    print(f"{name:<34} {py:>10.6f} {nd:>10.6f} {diff:>10.2e}")

assert max_diff < 1e-6, f"Node/Python mismatch: {max_diff}"
print(f"\nNode & Python forward passes agree (max diff {max_diff:.2e})")

os.remove(PROBES_FILE)
os.remove(NODE_SCRIPT)

probe                                  python       node       diff
established active trader            0.006457   0.006457   1.77e-07
normal user                          0.052413   0.052413   9.07e-08
fresh single tx                      0.153864   0.153864   1.23e-07
spam flooder                         0.283943   0.283943   4.23e-07

Node & Python forward passes agree (max diff 4.23e-07)
